# Lesson 53 · 🎨 MFCCs illustrated — watching an MFCC being born: waveform → spectrogram → mel spectrum → cepstral coefficients, layer by layer

**Goal**: actually draw how an MFCC comes into being: waveform → spectrogram → mel → cepstral coefficients, illustrated layer by layer.

> **🎨 The finale**: light on text, read it alongside the L50 pipeline; reading the plots is enough.

🔗 Aurora connection: `aurora.audio.mel.mel_spectrogram` · `aurora.audio.mfcc.mfcc` · `aurora.audio.mfcc.dct_ii`

← **Previous lesson** · [L52 · Audio Core complete](L52_features_done.ipynb)

> Last lesson covered **Audio Core complete**: wrapping up feature engineering, all 39 unit tests green, and assembling the evidence for interviews.
> This lesson turns **MFCCs into pictures**, so you can see every layer of the transformation.

## Learning goals

By the end of this lesson you should be able to:

1. Explain why the DCT-II can squeeze a smooth mel-spectrum envelope into the first few coefficients (the energy-compaction principle).
2. Say what physical quantity c[0], c[1] and c[2] each encode (overall energy, tilt, curvature).
3. Call `aurora.audio.mfcc.mfcc()` to produce a full MFCC matrix and show it as a heatmap.
4. Use `power_to_db` instead of a hand-written `log10`, keeping the implementation consistent with `mfcc.py`.

## 🌉 Background: cepstrum, Fourier transform, convolution (a quick start)

### What are the Fourier transform and its inverse?

You may have learned that **the Fourier transform turns a time signal into a frequency signal**. For example, a recording (the time domain) becomes "how much 440 Hz, how much 880 Hz, how much 1320 Hz is in this sound" (the frequency domain).

**The inverse Fourier transform goes the other way**: give me a "list of frequency components" and I can rebuild the original time signal.

---

### What is convolution, and why does it matter?

Picture the physical process when you speak:
1. **The excitation source** (the vocal folds in your larynx buzzing away, like a train of pulses)
2. **Vocal-tract filtering** (the shape of your larynx, pharynx and mouth boosts some frequencies and weakens others)

Mathematically: `recording = excitation * vocal tract` (`*` means convolution)

In the **frequency domain**, convolution has a magical property: `convolution becomes multiplication` ↔ `multiplication becomes convolution`

---

### The magic of logarithms: multiplication → addition

If `a = excitation, b = vocal-tract filter`, then a log turns `a × b` into:

$$\log(a \times b) = \log(a) + \log(b)$$

So **take the log of the audio spectrum and then apply a suitable transform, and you can split a "multiplicative mix" into an "additive decomposition"** — that's the core idea of the cepstrum.

---

### What is the cepstrum?

cepstrum = **the inverse Fourier transform of the log spectrum**

$$\text{cepstrum}[n] = \text{IDFT}(\log|\text{spectrum}|)$$

The effect:
- spectrum of the audio: `excitation × vocal tract` (multiplication)
- after the log: `log(excitation) + log(vocal tract)` (addition)
- after the IDFT: the coefficients split into two parts — the low orders correspond to the "vocal tract" (varying slowly across frequency), the high orders to the "excitation" (varying quickly)

**MFCC uses the DCT-II instead of the IDFT**, but the logic is exactly the same: both extract the "low-frequency content" of the spectrum.

---

### This lesson's storyline

L53 = **doing the cepstrum from the angle of "being able to compress the mel spectrum"**

We won't dig into convolution and decomposition; instead we focus on one practical fact:
> **adjacent mel bins are highly correlated (the spectral envelope is smooth) → the DCT can condense that "smoothness" into the first 13 coefficients → MFCC**

Let's see why the DCT can do this.

## 🔬 Code demo: how does the cepstrum actually "become code"? IFFT vs DCT-II

We said above that "cepstrum = the inverse Fourier transform of the log spectrum", but a formula alone is still abstract. Here we really run the code to see:

1. how the classic "cepstrum" algorithm is written, step by step (really using `np.fft.ifft`);
2. how it relates to, and differs from, the `dct_ii` that MFCC actually uses.

**Background story**: a human voice can roughly be seen as the convolution of the `excitation (vocal-fold vibration, a train of periodic pulses)` with the `vocal tract (the filter formed by throat and mouth)`. If we take the log of the spectrum of a stretch of speech and then do an inverse Fourier transform, the result has a very sharp "peak" — and its position corresponds exactly to the period of the speaker's fundamental (pitch). This is a classic speech-processing trick called "cepstral pitch detection". The code below does the trick live, and along the way shows how it relates to the DCT-II that MFCC really uses.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
# ═══ Code demo: really run the "cepstrum" IFFT(log|FFT(x)|), and compare with the DCT-II that MFCC uses ═══
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.mel import mel_spectrogram, power_to_db
from aurora.audio.mfcc import dct_ii

SR_demo2 = 22050
t2 = np.linspace(0, 0.5, int(SR_demo2 * 0.5), endpoint=False)

# Simulate a "voiced" sound with a 150 Hz fundamental: fundamental + 5 decaying overtones (the higher, the weaker)
f0 = 150
sig2 = sum(0.5 ** k * np.sin(2 * np.pi * f0 * (k + 1) * t2) for k in range(6))

# ① The classic "cepstrum" in three steps: FFT → log|·| → IFFT
spec = np.fft.fft(sig2 * np.hanning(len(sig2)))        # step 1: FFT of the whole signal (spectrum, complex)
log_mag = np.log(np.abs(spec) + 1e-10)                  # step 2: keep only the magnitude, then take the log (multiplication becomes addition)
real_cepstrum = np.fft.ifft(log_mag).real               # step 3: inverse Fourier transform of the log spectrum, keep the real part

# ② What MFCC actually does: mel filtering down to 40 dims → power_to_db → dct_ii
mel_s2 = mel_spectrogram(sig2, SR_demo2, n_fft=1024, n_mels=40)
log_mel2 = power_to_db(mel_s2[2], top_db=None)           # take frame 2, (40,)
mfcc_coeffs = dct_ii(log_mel2, norm="ortho")              # DCT-II, (40,)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

axes[0].plot(real_cepstrum[:300], color="#4C72B0")
peak_q = SR_demo2 / f0
axes[0].axvline(peak_q, color="red", linestyle="--", label=f"pitch period ≈ {peak_q:.0f} samples")
axes[0].set_title("① Classic cepstrum: IFFT(log|FFT(x)|)", fontsize=11)
axes[0].set_xlabel("quefrency (a \"pseudo-time\" index)")
axes[0].legend(fontsize=9)

axes[1].bar(range(40), mfcc_coeffs, color="#DD8452", width=0.8)
axes[1].set_title("② The MFCC way: DCT-II(log-mel)", fontsize=11)
axes[1].set_xlabel("DCT coefficient index k")

plt.tight_layout()
plt.savefig("v1_cepstrum_vs_dct.png", dpi=120)
plt.show()

print(f"① the classic cepstrum has a sharp peak at quefrency ≈ {peak_q:.0f} — that's the pitch period,")
print("   the classic way to separate the 'excitation (pitch)' from the 'vocal tract (timbre)' in a mixed signal.")
print()
print("② MFCC doesn't actually do an IFFT. Instead it first uses mel filters to squeeze the spectrum into 40 perceptually relevant bins")
print("   (a smoothing / dimensionality-reduction step in itself), then applies the DCT-II to that sequence, which is already real, non-negative and only 40 points long.")
print()
print("Why MFCC chooses the DCT-II rather than really doing an IFFT:")
print("  1) the DCT-II's input and output are both real — no complex numbers to handle and no manual .real as with the IFFT;")
print("  2) the DCT-II with norm='ortho' is naturally orthogonal — the inverse is just the transposed matrix, no inversion needed;")
print("  3) for a short (40-dim), smooth vector like the mel spectrum, the DCT-II costs far less than an IFFT over the whole spectrum.")
print("Before the TODO exercise below, real reconstruction-error numbers will back up these advantages.")


## Key intuition

**Smoothness of the spectrum → concentration of the coefficients**

A mel spectrum has 40–80 bins, and adjacent bins are highly correlated (the spectral envelope changes smoothly).
The DCT-II concentrates this "slow variation" into the first few coefficients, and the later ones are almost noise — that's why MFCC can describe timbre in 13 dimensions.

**The cepstral view** (optional reading): the spectrum is "excitation × vocal tract" (multiplication); after the log it becomes "log(excitation) + log(vocal tract)" (addition). Like the cepstrum, the DCT-II separates the two parts with an inverse transform; but we keep only the first 13 coefficients (the "vocal tract" part), which is enough to rebuild the timbre.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import chirp
from aurora.audio.mel import mel_spectrogram, power_to_db
from aurora.audio.mfcc import mfcc, dct_ii

## 🎯 Why the DCT-II? (and its formula in detail)

### What is the DCT?

The **discrete cosine transform (DCT)** is a way of breaking a signal down into "cosine waves". It sounds complicated, but the intuition is:

> **Like a "rainbow" splitting white light, the DCT splits a spectrum vector into a mix of a few "pure cosine waves"**.

How is it different from the Fourier transform?
- the **Fourier transform** uses `$e^{i\theta} = \cos(\theta) + i\sin(\theta)$` (complex sines and cosines)
- the **DCT** uses only cosines (real numbers); it's a "simplified" Fourier transform that's especially friendly to real signals

---

### The DCT-II formula, taken apart

```
y[k] = sum_{n=0}^{N-1}  S[n] * cos( pi/N * (n + 0.5) * k )
```

**Why does this formula achieve "energy compaction"?** Step by step:

**1. Why cosines?**

Because the cosine is the "smoothest" periodic function, the best fit for a "slowly varying" signal like a spectrum. Sines, by contrast, are 0 at the origin, and the high-frequency components tend to get mixed together.

**2. What does that `(n + 0.5)` mean?**

Imagine you have 40 mel bins (0 to 39). If you used `cos(pi/N * n * k)` directly, you'd waste half the information at n=0.

Changing it to `cos(pi/N * (n + 0.5) * k)` is like moving the sample points to **the center of each bin**:
- n=0 corresponds to center position 0.5
- n=1 corresponds to center position 1.5
- ...
- n=39 corresponds to center position 39.5

which uses the bins more efficiently.

**3. Why does the energy concentrate in the low orders?**

The key observation: the spectral envelope is **usually smooth** (adjacent bins have similar values).

And the cosine basis functions?
- k=0: `cos(0) = 1` (a constant) — captures "the overall average energy"
- k=1: `cos(pi/40 * (n+0.5) * 1)` — the slowest cosine, half a period stretched over all 40 bins — captures "a slow tilt from low to high"
- k=2: a faster cosine — captures a "small hill" shape
- k=13+: very fast cosines — can only capture "noisy jitter" (the spectrum itself doesn't have that much detail)

**For a smooth signal, the low-order (slow) basis functions fit 90% of it, and the high-order (fast) ones contribute next to nothing.** ✨

---

### Orthogonality of the DCT-II (a preview)

The DCT-II has another hugely important property: it's **orthogonal**, meaning the cosine basis functions are pairwise "perpendicular" (their inner products are 0).

What does that buy us?
- energy conservation: the original signal's total "energy" = the sum of squares of all the coefficients
- a simple inverse: multiply back by the same basis functions to recover the signal

We'll see this right away.

## 1. DCT compression: the energy-compaction principle

Each frame of the mel spectrum is a 40-dim vector `S[m]` (m = 0…39), and the DCT-II gives the coefficients:

```
y[k] = sum_{n=0}^{N-1}  S[n] * cos( pi/N * (n + 0.5) * k )
```

### What is "orthonormalization" (`norm="ortho"`)?

The formula above isn't normalized yet. With `norm="ortho"`, it's multiplied by some scaling factors:

$$c[k] = w_k \cdot \sum_{n=0}^{N-1} S[n] \cdot \cos\left(\frac{\pi}{N}(n+0.5)k\right)$$

where:
$$w_0 = \frac{1}{\sqrt{N}}, \quad w_k = \sqrt{\frac{2}{N}} \quad (k \geq 1)$$

**Why scale like this?**

#### The idea of an orthonormal basis (plain version)

Picture the three coordinate axes x, y, z in 3-D space:
- they're **mutually perpendicular** (perpendicular = orthogonal)
- they **all have length 1** (normalized)
- any vector can be described as "this much in x, this much in y, this much in z"

The DCT's cosine basis functions are the same: with `norm="ortho"`, they become **mutually perpendicular coordinate axes of length 1**.

#### What it buys us: energy conservation + a simple inverse

✨ **Energy conservation**: the original signal's total "energy" = `sum(S[n]²)` = the coefficients' total energy = `sum(c[k]²)`

✨ **A simple inverse**: multiply back by the same basis functions to recover `S = c @ B` (B is the basis matrix, one basis function per row — more on this before the TODO), no matrix inversion needed

---

### The constant basis function: why is c[0] the mean?

In particular, for `k=0`:

$$c[0] = \frac{1}{\sqrt{N}} \sum_{n=0}^{N-1} S[n] = \frac{1}{\sqrt{N}} \times (\text{mean} \times N) = \sqrt{N} \times \text{mean}(S)$$

So `c[0]` directly reflects the frame's **total log energy** (the factor √N keeps it in the same "energy units" as the other coefficients).

---

After orthonormalization, `k=0` is the constant basis function, `k=1` the slowest cosine, and so on.
Smooth spectral envelope → the high-order coefficients (large k) contribute very little → keeping the first 13 coefficients loses very little information.

## 🧪 Why must `power_to_db` come first? — a comparison experiment

The "Background" section mentioned in passing that "a log turns multiplication into addition". But what does that have to do with DCT compression? Why must the code do `power_to_db` first and then `dct_ii` — can't we just do `dct_ii(mel_spec)`?

**Intuition first**: the power values of a mel spectrum often differ enormously — a loud frequency component can be tens of thousands of times stronger than a quiet one. A shape like "one towering peak and almost flat ground everywhere else" is exactly what the DCT-II is worst at compressing (remember: the DCT relies on "a slowly varying, smooth spectrum" to concentrate energy into the low-order coefficients). Taking the log "flattens" those huge numerical gaps, turning the spectrum into a smoother curve, so the DCT can really make use of its "smooth → energy compaction" advantage.

Talk is cheap, so let's just run the comparison: DCT without the log, vs `power_to_db` first and then the DCT, and see how different the compression is.

In [ ]:
# ═══ Comparison: DCT directly without the log, vs power_to_db first then the DCT ═══
import numpy as np
from aurora.audio.io import chirp
from aurora.audio.mel import mel_spectrogram, power_to_db
from aurora.audio.mfcc import dct_ii

SR_pdb, N_MELS_pdb, N_KEEP_pdb = 22050, 40, 13
sig_pdb = chirp(duration=1.0, f0=200, f1=4000, sample_rate=SR_pdb)
mel_pdb = mel_spectrogram(sig_pdb, SR_pdb, n_fft=1024, n_mels=N_MELS_pdb)        # raw power spectrum, (n_frames, 40)
log_mel_pdb = power_to_db(mel_pdb, top_db=None)                                   # the log spectrum


def retention_rate(x, keep=N_KEEP_pdb):
    """Percentage of the total variance (energy) held by the first keep DCT coefficients."""
    coeffs = dct_ii(x, norm="ortho")
    var_per_k = np.var(coeffs, axis=0)
    return var_per_k[:keep].sum() / var_per_k.sum() * 100


retention_raw = retention_rate(mel_pdb)        # DCT directly on the raw power spectrum
retention_log = retention_rate(log_mel_pdb)    # log first, then the DCT

print(f"raw mel power spectrum value range: [{mel_pdb.min():.2e}, {mel_pdb.max():.2e}]  ← max/min ≈ {mel_pdb.max()/max(mel_pdb.min(), 1e-12):.0e}×")
print(f"log-mel (dB) value range:           [{log_mel_pdb.min():.1f}, {log_mel_pdb.max():.1f}]  ← only a few tens apart")
print()
print(f"❌ no log, straight dct_ii(mel_spec): first {N_KEEP_pdb} coefficients keep {retention_raw:.1f}% of the energy")
print(f"✅ power_to_db first, then dct_ii:    first {N_KEEP_pdb} coefficients keep {retention_log:.1f}% of the energy")
assert retention_log > retention_raw, "compression should be better after taking the log"
print()
print("Conclusion: without the log, the values are dominated by a few 'peak' bins, and the DCT needs more coefficients to describe the ups and downs,")
print("      so compression is much worse; taking the log first flattens the gaps, and only then can the DCT condense 40 dims into 13 effectively.")
print("      This also matches how the ear works — loudness is perceived on a log scale (decibels), not linearly, so the log makes the features fit hearing better.")


In [ ]:
SR = 22050
N_MELS = 40

# Synthesize a mixed tone: 440 Hz fundamental + overtones, imitating a vowel timbre
t = np.linspace(0, 1.0, SR, endpoint=False)
signal = (
    0.6 * np.sin(2 * np.pi * 440 * t)
    + 0.3 * np.sin(2 * np.pi * 880 * t)
    + 0.1 * np.sin(2 * np.pi * 1320 * t)
)

# Take a single frame of mel spectrum (frame 10, away from the first frames affected by center padding)
mel_spec = mel_spectrogram(signal, SR, n_fft=1024, n_mels=N_MELS)
log_mel_frame = power_to_db(mel_spec[10], top_db=None)   # single-frame log-mel, shape (40,) — consistent with mfcc.py

# DCT transform
dct_all = dct_ii(log_mel_frame, norm="ortho")          # shape (40,)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].bar(range(N_MELS), log_mel_frame, color="#4C72B0", width=0.8)
axes[0].set_title("Log-mel spectrum (one frame, 40 dims)", fontsize=12)
axes[0].set_xlabel("mel bin index")
axes[0].set_ylabel("dB")

axes[1].bar(range(N_MELS), dct_all, color="#DD8452", width=0.8)
axes[1].axvline(12.5, color="red", linestyle="--", linewidth=1.5, label="cutoff k=13")
axes[1].set_title("DCT-II output (40 coefficients)", fontsize=12)
axes[1].set_xlabel("DCT coefficient index k")
axes[1].set_ylabel("coefficient value")
axes[1].legend()

plt.tight_layout()
plt.savefig("v1_dct_compression.png", dpi=120)
plt.show()
print("✅ What to see: the energy is concentrated on the left, k < 13; the coefficients on the right are close to 0")

In [ ]:
# ─ Visualizing the DCT-II basis functions: why does energy compaction happen? ─────
N_MELS = 40
k_arr = np.arange(N_MELS)
n_arr = np.arange(N_MELS)

# Build a few DCT-II basis functions
fig, axes = plt.subplots(4, 1, figsize=(12, 6))

basis_k = [0, 1, 2, 12]  # basis functions of order 0, 1, 2 and 12

for ax, k in zip(axes, basis_k):
    # The cosine basis before orthonormalization
    basis_raw = np.cos(np.pi / N_MELS * (n_arr + 0.5) * k)
    ax.plot(n_arr, basis_raw, 'o-', color="#4C72B0", markersize=4, linewidth=1.5, label=f'k={k} (cosine)')
    ax.fill_between(n_arr, 0, basis_raw, alpha=0.3, color="#4C72B0")
    ax.set_ylabel(f'c[{k}]', fontsize=10)
    ax.set_title(
        f'DCT-II basis function k={k}: ' +
        (['constant (total energy)', 'slow tilt (low frequency)', 'gentle curvature', 'high-frequency jitter'][k if k < 3 else 3]),
        fontsize=11
    )
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.5, N_MELS - 0.5)

axes[-1].set_xlabel('mel bin index n')
fig.suptitle('The DCT-II cosine basis functions: why is there lots of energy in the low orders and noise in the high ones?', fontsize=12, y=1.00)
plt.tight_layout()
plt.savefig("v1_dct_basis.png", dpi=120)
plt.show()

print(f"✅ What to see: k=0 is flat (a constant), k=1 undulates slowly, k=12 oscillates fast.")
print(f"   A smooth spectrum can be fitted with the first few basis functions; the high-order coefficients are close to 0.")


## 🧩 Seeing is believing: can you really "build" a spectrum curve out of cosines?

Above we drew individual basis functions, but claims like "c[1] encodes tilt, c[2] encodes curvature" are just words in the end. So let's do something concrete: multiply each coefficient back onto its basis function, add them up, and see how close the reconstructed curve comes to the real log-mel spectrum. This is also the most direct answer to what the "spectral envelope" is: it's the smooth curve connecting the values of the 40 mel bins, and the DCT approximates that curve with a handful of cosine waves.

In [ ]:
# ═══ Demo: c[0]·basis 0 + c[1]·basis 1 + c[2]·basis 2 + ... gradually builds up the real spectrum ═══
n_arr = np.arange(N_MELS)
w0_fit = 1.0 / np.sqrt(N_MELS)
wk_fit = np.sqrt(2.0 / N_MELS)


def basis_k(k):
    """The k-th DCT-II basis function (with ortho scaling), length N_MELS."""
    raw = np.cos(np.pi / N_MELS * (n_arr + 0.5) * k)
    return raw * (w0_fit if k == 0 else wk_fit)


# log_mel_frame and dct_all come from the earlier "DCT compression" cell (same frame, reused directly)
approx_orders = [0, 1, 2, 3, 12, 39]  # sum the coefficients up to this order

fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharey=True)
for ax, K in zip(axes.flat, approx_orders):
    recon = sum(dct_all[k] * basis_k(k) for k in range(K + 1))
    ax.plot(n_arr, log_mel_frame, color=plt.rcParams["grid.color"], lw=2.5, label="real spectrum")
    ax.plot(n_arr, recon, color="#DD8452", lw=1.8, ls="--", label=f"summed up to k={K}")
    ax.set_title(f"fit with only the first {K + 1} coefficients", fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel("mel bin index n")

fig.suptitle("Adding c[0], c[1], c[2]... one at a time: how does the curve close in on the real spectrum?", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("v1_basis_fit_demo.png", dpi=120)
plt.show()

print("Look at k=0: just a horizontal line — it captures only the 'overall volume' (matching c[0] = total log energy).")
print("Look at k=1: with the first slow cosine added, the curve starts to 'tilt' — c[1] controls exactly this overall tilt (positive/negative means low/high frequencies dominate).")
print("Look at k=2: add a slightly more curved cosine and the curve gets an 'arch' — c[2] controls this rough curvature (bulging up or dipping down).")
print("Look at k=12 vs k=39: by k=12 the curve almost coincides with the real spectrum; going on to k=39 changes almost nothing —")
print("      direct evidence that 'the high-order coefficients carry almost no information', and the reason MFCC keeps only the first 13 dims.")
print()
print("The 'spectral envelope' is exactly this smooth curve connecting the values of the 40 mel bins; what the DCT does")
print("      is approximate that curve with a few cosine waves of different 'curviness'.")


## 📐 Geometric intuition for what the coefficients mean

Each DCT coefficient `c[k]` measures "how similar the input is to the k-th cosine basis function". Different k correspond to different cosine frequencies:

| k | Basis shape | Effect on the mel spectrum | Physical meaning |
|------|----------|---------|---------|
| k=0 | constant (flat) | overall average | **total log energy**: c[0] = √N × mean(S); large = loud |
| k=1 | half-period cosine (slow) | low-frequency bins weighted ↑, high-frequency ↓ | **spectral tilt**: c[1] > 0 = low frequencies dominate, c[1] < 0 = high frequencies dominate |
| k=2 | one-period cosine (a bit faster) | high at both ends, low in the middle (a bowl) | **spectral curvature**: c[2] < 0 = an arch with a mid-band hump, like a mid-frequency resonance (e.g. the /a/ sound) |
| k=3..12 | higher-frequency cosines (fast) | increasingly fine formant structure | **fine-grained formants**: c[3..12] differ most between vowels |
| k≥13 | high-frequency jitter (very fast) | captures only noisy detail, useless for sound quality | **noise**: usually dropped |

(Want to actually "see" the "tilt" and "curvature" this table talks about? Scroll back one cell — there, c[0], c[1], c[2] are added back step by step on real spectrum data, with the fit drawn out.)

### A little acoustic background first: what is "resonance"?

Ever blown across the mouth of an empty glass bottle and heard it go "hooo——"? The bottle itself makes no sound, but the air column inside gets "amplified" at one particular frequency into a pitch — that's **resonance**: a cavity amplifies vibrations at some frequencies and damps others. The bigger/wider the bottle, the lower its resonant frequency; change the shape and the resonant frequency changes with it.

Your vocal tract (throat, mouth, nasal cavity) is a "bottle" whose shape keeps changing. As the sound from the vocal folds (the excitation, with lots of frequency components) passes through the tract, a few particular frequencies get amplified by its shape — those amplified frequency peaks are called **formants**. Move your tongue, lips or jaw, and the tract's shape changes, and the formants move with it — which is the physical reason the same voice can "shape" different vowels (/a/, /i/, /u/...).

What exactly is a **formant**? In vowels, vocal-tract resonance reinforces certain frequencies. For example, the /a/ sound has 3 formants (roughly 730, 1090 and 2440 Hz), and /i/ has them in different places. The DCT's mid-order coefficients (3–12) encode these individual structures.

---

## 2. What each order of coefficient means physically

| Coefficient | Basis shape | Physical meaning |
|------|-----------|---------|
| `c[0]` | constant (all ones) | the mean of the log-mel spectrum = the frame's total log energy |
| `c[1]` | half-period cosine | the low-frequency tilt of the spectral envelope (high- vs low-frequency energy) |
| `c[2]` | one-period cosine | the "bow-shaped" curvature of the envelope |
| `c[3..12]` | higher-frequency cosines | increasingly fine formant structure |
| `c[13+]` | high-frequency jitter | noise, usually dropped |

Key formula (`norm="ortho"`, `k=0`):

```
c[0] = sqrt(1/N) * sum_{n} S[n]   # i.e. the mean of the N mel bins (times sqrt(N))
```

c[0] changes with volume, so for speaker-independent recognition it's often dropped or removed with **CMN (cepstral mean normalization)**.

The idea of CMN: c[0] reflects the overall volume, and volume varies from person to person and setting to setting (different microphones, different distances). If you train a speaker-recognition model without removing c[0], the model learns "volume features" rather than "voiceprint features". What CMN does is subtract c[0]'s mean over the whole utterance, bringing c[0] to zero and forcing the model to pay attention to the other coefficients (c[1], c[2] and so on).

**Understanding CMN through the "bottle" analogy**: with the same bottle, you can blow softly or hard, and the volume of the sound changes, but the bottle's resonant frequency (the key feature deciding "which sound it is") doesn't — volume is just "how hard you blow", not information about the bottle's "shape". c[0] is like "how hard you blow": it reflects volume, but not "the shape of the vocal tract" (that is, not the timbre itself). What CMN does is wipe out the differences in "how hard", keeping only the "bottle shape" (vocal-tract) information, so the model isn't led astray by irrelevant factors like how loudly someone speaks or how far away the microphone is.

In [ ]:
# Compare three timbres: pure low tone vs pure high tone vs a mix of the two
t = np.linspace(0, 1.0, SR, endpoint=False)

sig_low  = np.sin(2 * np.pi * 300 * t)                          # pure low frequency
sig_high = np.sin(2 * np.pi * 3000 * t)                         # pure high frequency
sig_mix  = 0.5 * sig_low + 0.5 * sig_high                       # low + high

signals  = [sig_low, sig_high, sig_mix]
labels   = ["pure low 300 Hz", "pure high 3000 Hz", "low + high mix"]
colors   = ["#4C72B0", "#DD8452", "#55A868"]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)

for ax, sig, label, color in zip(axes, signals, labels, colors):
    mel_s = mel_spectrogram(sig, SR, n_fft=1024, n_mels=N_MELS)
    lm = power_to_db(mel_s[10], top_db=None)
    c = dct_ii(lm, norm="ortho")[:13]
    ax.bar(range(13), c, color=color, width=0.8)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("MFCC coefficient k")
    if ax is axes[0]:
        ax.set_ylabel("coefficient value")
    # Label c0
    ax.annotate(f"c[0]={c[0]:.1f}", xy=(0, c[0]), xytext=(2, c[0]+1),
                arrowprops=dict(arrowstyle="->", color=plt.rcParams["grid.color"]), fontsize=9)

fig.suptitle("The first 13 MFCC coefficients for different timbres", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("v1_mfcc_timbre.png", dpi=120)
plt.show()
print("✅ What to see: c[0] reflects total energy; c[1] positive → low frequencies dominate, c[1] negative → high frequencies dominate")

## 🔋 What are "energy" and "variance"?

In signal processing, **energy** is usually defined as:

$$E = \sum_{n=0}^{N-1} x[n]^2$$

which is similar to the idea of "kinetic energy" `(1/2)mv²` in physics — the bigger the values, the more "energy".

### Why use "variance" to measure how important a coefficient is?

Over many frames of audio, each DCT coefficient changes along the time axis. For example:
- in **silent frames**, c[0] and c[1] are both small
- in **loud frames**, c[0] and c[1] are both large

**Variance** = how much a coefficient "swings" from frame to frame:

$$\text{var}(c[k]) = \frac{1}{T} \sum_{t=0}^{T-1} (c[k,t] - \text{mean}(c[k]))^2$$

**Why does large variance = an important coefficient?**

- **large variance** → the coefficient changes noticeably between frames → it carries rich **timbre information** (telling different phonemes apart)
- **small variance** → the coefficient is nearly constant across all frames → redundant **noise information** (the same for every sound)

From a machine-learning angle: if you're training a speaker-recognition model, the high-variance coefficients help the model tell voices apart better, so they're more "important".

### Taking the variance over "many frames": the power of broadcasting

The code has:
```python
log_mel_all.shape = (n_frames, 40)  # e.g. (100 frames, 40 mel bins)
dct_all_frames.shape = (100, 40)
var_per_bin = np.var(dct_all_frames, axis=0)  # axis=0 means variance across time
# result: var_per_bin.shape = (40,)  # one variance per coefficient
```

What `axis=0` means: **collapse along the time axis** — for each column (each coefficient index k), compute how much it fluctuates over time.

---

## Parameter experiment: quantifying energy compaction

**Parameters**: `N_MELS = 40`, keeping `n_mfcc = 13`

**What to expect**:
- the variance summed over the first 13 coefficients >> the variance summed over the remaining 27
- the energy retention rate (power of the first 13 dims / all 40 dims) should be > 95%

Change `N_MELS` to 80, or change how complex the signal is (chirp vs pure tone), and watch how the energy retention rate changes.

## 📦 Interlude: what is NumPy "broadcasting"?

The data we're about to process is an `(n_frames, 40)` 2-D array — each row is one frame's log-mel spectrum. The code will simply write `dct_ii(log_mel_all, norm="ortho")`, without a `for` loop going frame by frame. How does that work?

**If you only know `for` loops**, you might write:

```python
results = []
for row in log_mel_all:          # process one row at a time
    results.append(dct_ii(row, norm="ortho"))
results = np.array(results)
```

NumPy offers a faster, neater way, and the core trick is called **broadcasting**: arrays of "different shapes" can be combined directly, with NumPy automatically lining up and repeating dimensions, so you don't write the loop yourself.

### A small example: add the same vector to every row of a 2-D matrix

This example has nothing to do with the DCT; it's just to build intuition:

```python
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])          # shape (3, 3)
v = np.array([10, 100, 1000])       # shape (3,)

print(M + v)
# NumPy automatically "copies" v into 3 rows, lines each one up with a row of M, then adds elementwise
# no for loop needed, and no manual np.tile(v, (3, 1)) either
```

### The `axis` argument: telling NumPy which direction to "collapse"

```python
print("sum along rows (axis=1):", M.sum(axis=1))   # add up each row, result shape (3,)
print("sum down columns (axis=0):", M.sum(axis=0))   # add up each column, result shape (3,)
```

Mnemonic: **`axis=0` operates "across rows" (the row dimension disappears from the result); `axis=1` operates "across columns" (the column dimension disappears from the result)**. The "energy and variance" section later uses `np.var(dct_all_frames, axis=0)`, which means "variance across frames (across rows)" — stack many frames together, look at how each coefficient fluctuates between frames, and you're left with the 40 coefficients' individual variances.

### How does `dct_ii` "automatically" process row by row?

Open the source of `aurora.audio.mfcc.dct_ii`, and the key line is:

```python
y = x @ basis.T   # x: (..., N), basis.T: (N, N)
```

`@` is matrix multiplication. If `x` is 1-D (one frame, shape `(40,)`), this is "vector × matrix", and the result is still 1-D. If `x` is 2-D (many frames, shape `(n_frames, 40)`), matrix multiplication automatically treats it as "many row vectors, each multiplied by the same matrix" — every row gets the same transform independently, and the `for` loop is "bundled" into one matrix product.

The very next line, `y = y * scale` (`scale` has shape `(40,)`), is standard broadcasting: an `(n_frames, 40)` result times a `(40,)` vector, with NumPy automatically "spreading" `scale` across every row and multiplying elementwise.

**One-sentence summary**: whether the input is one frame or ten thousand, the code inside `dct_ii` doesn't change at all — that's the power of matrix multiplication (handling "row by row") working together with broadcasting (handling "elementwise scaling"). A small experiment below confirms it.

In [ ]:
# ═══ Check: a hand-written for loop vs dct_ii on a 2-D array directly — exactly the same result ═══
from aurora.audio.mfcc import dct_ii

demo_frames = np.random.default_rng(0).standard_normal((5, 40))  # pretend: 5 frames, 40 dims

loop_result = np.array([dct_ii(row, norm="ortho") for row in demo_frames])
vectorized_result = dct_ii(demo_frames, norm="ortho")

print("for-loop result shape:         ", loop_result.shape)
print("result shape passing a 2-D array:", vectorized_result.shape)
print("are the two exactly the same:   ", np.allclose(loop_result, vectorized_result))


In [ ]:
# Note: this code shows off NumPy's "broadcasting" ability
# log_mel_all is an (n_frames, N_MELS) 2-D array
# dct_ii() applies the DCT to "each row" independently; the result is still (n_frames, N_MELS)
# batch processing like this does away with explicit for loops, one of NumPy's core strengths

SR = 22050
N_MELS = 40
N_KEEP = 13

# Use a chirp signal (a richer spectrum, harder to compress)
t = np.linspace(0, 1.0, SR, endpoint=False)
sig_chirp = chirp(duration=1.0, f0=200, f1=4000, sample_rate=SR)

mel_s = mel_spectrogram(sig_chirp, SR, n_fft=1024, n_mels=N_MELS)

# DCT every frame, and compute each bin's variance
log_mel_all = power_to_db(mel_s, top_db=None)   # (n_frames, N_MELS) — consistent with mfcc.py
dct_all_frames = dct_ii(log_mel_all, norm="ortho")  # (n_frames, N_MELS)

var_per_bin = np.var(dct_all_frames, axis=0)  # (N_MELS,)
power_total  = var_per_bin.sum()
power_kept   = var_per_bin[:N_KEEP].sum()
retention    = power_kept / power_total * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# Variance of each coefficient
axes[0].bar(range(N_MELS), var_per_bin, color="#4C72B0", width=0.8)
axes[0].axvline(N_KEEP - 0.5, color="red", linestyle="--", lw=1.5, label=f"cutoff k={N_KEEP}")
axes[0].set_title("DCT coefficient variance (across frames)", fontsize=12)
axes[0].set_xlabel("DCT coefficient index k")
axes[0].set_ylabel("variance")
axes[0].legend()

# Cumulative energy retention
cumulative = np.cumsum(var_per_bin) / power_total * 100
axes[1].plot(range(N_MELS), cumulative, marker="o", markersize=4, color="#DD8452")
axes[1].axhline(95, color=plt.rcParams["grid.color"], linestyle=":", lw=1.5, label="95% threshold")
axes[1].axvline(N_KEEP - 1, color="red", linestyle="--", lw=1.5, label=f"k={N_KEEP-1}")
axes[1].set_title("Cumulative variance retained (%)", fontsize=12)
axes[1].set_xlabel("keep the first k coefficients")
axes[1].set_ylabel("retained (%)")
axes[1].legend()

plt.tight_layout()
plt.savefig("v1_energy_retention.png", dpi=120)
plt.show()

print(f"✅ N_MELS={N_MELS}, keeping the first {N_KEEP} dims:")
print(f"   variance summed over the first {N_KEEP} coefficients = {power_kept:.4f}")
print(f"   variance summed over the last {N_MELS-N_KEEP} coefficients = {power_total - power_kept:.4f}")
print(f"   energy retention = {retention:.1f}%")
assert retention > 80.0, f"energy retention {retention:.1f}% is lower than expected (should be > 80%)"


In [ ]:
# Full MFCC chain demo: an mfcc() heatmap
# The full chain: mel_spectrogram -> power_to_db -> dct_ii -> [:, :n_mfcc]
SR_demo = 22050
N_MFCC = 13

# A chirp signal: its frequency changes over time, which makes for a more representative MFCC heatmap
sig_demo = chirp(duration=2.0, f0=200, f1=4000, sample_rate=SR_demo)

# Call aurora's complete mfcc() function
mfcc_matrix = mfcc(sig_demo, SR_demo, n_mfcc=N_MFCC, n_fft=1024, n_mels=40)  # (n_frames, 13)

print(f'MFCC matrix shape: {mfcc_matrix.shape}  -> (n_frames, n_mfcc)')

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(
    mfcc_matrix.T,           # transposed: y-axis = coefficient index, x-axis = time frame
    aspect='auto',
    origin='lower',
    interpolation='nearest',
    cmap='RdBu_r'
)
ax.set_title('MFCC heatmap (chirp signal, end-to-end mfcc() output)', fontsize=13)
ax.set_xlabel('time frame')
ax.set_ylabel('MFCC coefficient index')
ax.set_yticks(range(N_MFCC))
ax.set_yticklabels([f'c[{k}]' for k in range(N_MFCC)], fontsize=8)
fig.colorbar(im, ax=ax, label='coefficient value (dB domain)')
plt.tight_layout()
plt.show()
print('✅ full MFCC chain checked: the end-to-end mfcc() output is visualized')

## 📏 Let the numbers talk: DCT-II reconstruction vs "just doing a DFT/IFFT" — how different are the errors?

The cepstrum demo earlier explained "why MFCC chooses the DCT-II rather than really doing an IFFT"; here we check it with concrete reconstruction-error numbers. The upcoming TODO exercise is the "by hand" version of exactly this experiment, so first have a look at what the whole process looks like.

**Experiment design**: take the same frame of log-mel spectrum (40 dims), compress it to "about 13 numbers" with each of two methods, reconstruct back to 40 dims, and see whose reconstruction error is smaller:
- **Method 1 (DCT-II, the MFCC way)**: forward DCT-II to get 40 coefficients, keep only the first 13, zero the rest, then inverse-transform.
- **Method 2 (really doing a DFT/IFFT)**: forward real FFT (`np.fft.rfft`), keep only the first 7 complex bins (1 DC + the real and imaginary parts of 6 complex bins ≈ 13 degrees of freedom), zero the rest, then `np.fft.irfft` back to 40 dims.

The two methods keep roughly the same number of "degrees of freedom" (independent numbers stored), so the comparison is fair.

In [ ]:
# ═══ Quantitative comparison: truncated DCT-II reconstruction vs truncated DFT/IFFT reconstruction — whose error is smaller? ═══
SR_cmp, N_MELS_cmp, N_KEEP_cmp = 22050, 40, 13
sig_cmp = chirp(duration=1.0, f0=200, f1=4000, sample_rate=SR_cmp)
mel_cmp = mel_spectrogram(sig_cmp, SR_cmp, n_fft=1024, n_mels=N_MELS_cmp)
log_mel_cmp = power_to_db(mel_cmp, top_db=None)   # (n_frames, 40)

# Build the orthonormal DCT-II basis matrix D by hand (exactly the same formula as in the dct_ii source; computed here so you can compare)
k_arr = np.arange(N_MELS_cmp)
n_arr = np.arange(N_MELS_cmp)
D_cmp = np.cos(np.pi / N_MELS_cmp * np.outer(k_arr, n_arr + 0.5))
D_cmp[0] *= 1.0 / np.sqrt(N_MELS_cmp)
D_cmp[1:] *= np.sqrt(2.0 / N_MELS_cmp)

errs_dct, errs_dft = [], []
for idx in range(5, 30):  # gather statistics over 25 consecutive frames, so a single frame can't fluke it
    frame = log_mel_cmp[idx]

    # Method 1: truncated DCT-II reconstruction (orthogonal matrix: the transpose is the inverse, D.T == D^{-1})
    c_full = D_cmp @ frame
    c_trunc = c_full.copy()
    c_trunc[N_KEEP_cmp:] = 0.0
    recon_dct = D_cmp.T @ c_trunc
    errs_dct.append(np.sqrt(np.mean((recon_dct - frame) ** 2)))

    # Method 2: really doing a truncated DFT/IFFT reconstruction (keeping about the same degrees of freedom)
    F_full = np.fft.rfft(frame)
    F_trunc = F_full.copy()
    F_trunc[7:] = 0.0                                     # 1 DC + 6 complex bins ≈ 13 degrees of freedom
    recon_dft = np.fft.irfft(F_trunc, n=N_MELS_cmp)
    errs_dft.append(np.sqrt(np.mean((recon_dft - frame) ** 2)))

print(f"truncated DCT-II reconstruction, mean RMS error = {np.mean(errs_dct):.2f} dB")
print(f"truncated DFT/IFFT reconstruction, mean RMS error = {np.mean(errs_dft):.2f} dB")
assert np.mean(errs_dct) < np.mean(errs_dft), "the DCT-II should have a smaller error than a direct DFT/IFFT"
print()
print("Why is the DCT-II error smaller? The key is the 'boundary':")
print("The DFT implicitly assumes the signal 'wraps around and repeats periodically' — if the log-mel spectrum's value 0 and value 39 differ a lot,")
print("joining them end to end creates a 'cliff', the DFT spends lots of high-frequency coefficients describing that cliff, and truncation leaves a big error.")
print("The DCT-II's (n+0.5) amounts to 'mirroring' the signal symmetrically before expanding it, so the ends join smoothly, with no cliff,")
print("and a few low-order coefficients describe the whole curve well — the core reason MFCC chooses the DCT-II.")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(["DCT-II\n(used by MFCC)", "DFT/IFFT\n(done directly)"], [np.mean(errs_dct), np.mean(errs_dft)],
       color=["#4C72B0", "#DD8452"])
ax.set_ylabel("mean RMS reconstruction error (dB)")
ax.set_title("Keeping the same ~13 degrees of freedom, the DCT-II reconstructs with less error")
plt.tight_layout()
plt.savefig("v1_dct_vs_dft_recon_error.png", dpi=120)
plt.show()


## 📚 Orthogonal matrices and the inverse transform (getting ready for the TODO)

### What is an "orthonormal basis matrix"?

Say you have N DCT basis functions (each a vector of length N). Stack them into a matrix:

$$D = \begin{bmatrix}
\text{basis function 0} \\
\text{basis function 1} \\
\vdots \\
\text{basis function N-1}
\end{bmatrix} \quad \text{shape: (N, N)}$$

Each row is one basis function, for example:
- row 0: `[cos(0), cos(0), ..., cos(0)]` = `[1, 1, ..., 1]` (a constant)
- row 1: `[cos(π/N×0.5), cos(π/N×1.5), ..., cos(π/N×(N-0.5))]` (a slow cosine)
- ...

What does **"orthogonal"** mean? The inner product of any two different rows is 0 (they're "perpendicular").
What does **"normalized"** mean? The inner product of each row with itself is 1 (it has "length 1").

### A simple example: the 2-D coordinate system

Picture the coordinate system of 2-D space, with the two basis vectors:
$$\mathbf{e}_x = \begin{bmatrix} 1 \\ 0 \end{bmatrix}, \quad \mathbf{e}_y = \begin{bmatrix} 0 \\ 1 \end{bmatrix}$$

Together they form the matrix:
$$\mathbf{I} = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$

Property: `I @ I.T = I` (orthogonality)

Any point's coordinates can be written as `(x, y) = x·e_x + y·e_y`

Conversely, given coordinates `(a, b)`, how do you recover `x` and `y`?
$$\begin{bmatrix} x \\ y \end{bmatrix} = \mathbf{I}^{-1} \begin{bmatrix} a \\ b \end{bmatrix} = \begin{bmatrix} a \\ b \end{bmatrix}$$

**Orthogonal matrices** have a superpower: `Q.T = Q^{-1}` (the transpose is the inverse)!

---

### The inverse of the DCT-II

The DCT-II basis matrix D is also **orthogonal** (with `norm='ortho'`).

**Forward transform**: signal S (length N) → coefficients c
$$\mathbf{c} = \mathbf{D} \mathbf{S}^T$$

(D is an (N, N) matrix; S is a row vector, which becomes a column vector after transposing)

**Inverse transform** (one frame; S is a row vector of length N, written `S^T` to make it a column vector that multiplies nicely with the matrix):
$$\mathbf{S}^T = \mathbf{D}^T \mathbf{c}$$

Derivation: because D is orthogonal, `D^{-1} = D^T`. The forward transform is `c = D S^T`; **left-multiply** both sides by `D^T`:
$$\mathbf{D}^T \mathbf{c} = \mathbf{D}^T \mathbf{D}\, \mathbf{S}^T = \mathbf{S}^T$$
(`D^T D = I`, which cancels exactly the D on the right, leaving only `S^T`) — and that's where `D_cmp.T @ c_trunc` in the "let the numbers talk" code came from.

---

**But the TODO handles a whole batch of frames**: `dct_truncated.shape = (n_frames, 40)`, with one frame per **row**, which isn't the "S is a column vector" convention above. `dct_ii` is implemented internally with "a row is a frame" (the source says `y = x @ basis.T`), so the batched forward transform is:

$$\mathbf{c} = \mathbf{S}\, \mathbf{D}^T \quad (\text{each row of S is one frame})$$

Again using the orthogonality of `D`: this time S stands on the **left** of the multiplication, so we add D on the **right** to cancel the `D^T` on the right (**right-multiply** both sides by `D`):
$$\mathbf{c}\,\mathbf{D} = \mathbf{S}\,\mathbf{D}^T \mathbf{D} = \mathbf{S}$$
(`D^T D = I`; this time it's the `D^T` on the right that cancels)

In other words: **for many frames (one frame per row), the inverse transform is `c @ D`, not `c @ D.T`!**

**Why left-multiply by `D^T` in one case and right-multiply by `D` in the other?** It's decided purely by "which side of the multiplication S stands on" — with S on the right (a column vector), add `D^T` on the left to cancel the D on the left; with S on the left (a row vector), add `D` on the right to cancel the `D^T` on the right. Matrix multiplication doesn't commute, so the direction of "cancelling" has to switch sides along with S, but the matrix used is **always the same D** (or its transpose). That's also the key to answering "why right-multiply `@ B` rather than left-multiply": in the TODO, `dct_truncated` stands on the left of the multiplication, so we right-multiply.

Or, with the truncated coefficients c_trunc (high orders zeroed):
$$\mathbf{S}_{approx} = \mathbf{c}_{trunc} \cdot \mathbf{D}$$

This gives an **approximate reconstruction** (because the high-frequency coefficients were thrown away).

---

### The matrix multiplication in the TODO

The TODO code ends up as `log_mel_recon = dct_truncated @ B` (**no** `.T`!), where:
- `dct_truncated.shape = (n_frames, 40)`: the first 13 columns are DCT coefficients, the last 27 are 0
- `B.shape = (40, 40)`
- the result `log_mel_recon.shape = (n_frames, 40)`

**A direct answer to the question people most often get stuck on: is B just D itself, or some variation of D?**

**Answer: B is D itself, built with exactly the same formula** — `B[k, n] = cos(pi/N*(n+0.5)*k)`, multiplied by exactly the same ortho scaling as before (`w0`, `wk`). No need to cook up another matrix: the basis matrix you saw in the "Why the DCT-II" section, `D_cmp` in the "let the numbers talk" comparison, and the `B` the TODO asks you to build are **the same matrix**, just under different names.

```python
# Forward DCT-II matrix (not yet orthonormalized)
k = np.arange(40)
n = np.arange(40)
D_raw = np.cos(np.pi / 40 * np.outer(k, n + 0.5))  # (40, 40)
# D_raw[k, n] = cos(pi/40 * (n+0.5) * k)

# Add the ortho scaling
w0 = 1.0 / np.sqrt(40)
wk = np.sqrt(2.0 / 40)
D = D_raw.copy()
D[0, :] *= w0
D[1:, :] *= wk

# B is D, unchanged; for batched data (many frames, one per row) the inverse is a right-multiplication by B:
# log_mel_recon = dct_truncated @ B     ← matches the "c @ D" derived above; no extra transpose needed
```

In the TODO, B is exactly this matrix — corresponding exactly to "inverse transform for many frames = `c @ D`" derived above: right-multiply by `B` directly, no `.T` needed.

---



In [ ]:
# ── TODO exercise: DCT reconstruction error ────────────────────────────────────────────
# Goal: reconstruct the log-mel spectrum from the first N_KEEP MFCC coefficients and quantify the information lost.
#
# Orthogonality of the DCT-II: an orthogonal matrix satisfies D @ D.T = I, so D.T = D^{-1}
# Multiplying the first N_KEEP coefficients (the rest zeroed) by the basis matrix approximately recovers the original vector.
#
# Steps:
#   1. truncate the high-order coefficients of dct_all: dct_truncated[:, N_KEEP:] = 0
#   2. build the orthonormal DCT-II basis matrix B, shape (N_MELS, N_MELS)
#      B[k, n] = cos(pi/N_MELS * (n+0.5) * k), with ortho scaling applied
#   3. inverse transform: log_mel_recon = dct_truncated @ B (note: no extra transpose!)
#      Why not B.T: dct_ii computes c = S @ B.T internally, with "a row is a frame";
#      B is orthogonal (B @ B.T = I), so multiplying both sides by the inverse of B.T, (B.T)^{-1} = B,
#      gives S = c @ B — right-multiply by B directly, not B.T.
#   4. compute the RMS error and check it is < 8 dB

SR = 22050
N_MELS = 40
N_KEEP = 13

sig_todo = chirp(duration=1.0, f0=200, f1=4000, sample_rate=SR)
mel_todo = mel_spectrogram(sig_todo, SR, n_fft=1024, n_mels=N_MELS)
log_mel_todo = power_to_db(mel_todo, top_db=None)   # (n_frames, N_MELS)
dct_todo = dct_ii(log_mel_todo, norm='ortho')        # (n_frames, N_MELS)

# === Implement below ===

# 1. Truncate the coefficients (keep the first N_KEEP columns, zero the rest)
dct_truncated = dct_todo.copy()
dct_truncated[:, N_KEEP:] = 0.0

# 2. Build the orthonormal DCT-II basis matrix B
#    Hints:
#    - k and n both run over 0..N_MELS-1
#    - build the unscaled cos matrix first
#    - then apply the ortho scaling: w0 = 1/sqrt(N_MELS) on row 0, wk = sqrt(2/N_MELS) on the other rows
#    - that makes B an orthogonal matrix, and B.T the inverse-transform matrix
#
#    A code skeleton for reference (note: it's only a hint; you need to write it out in full yourself):
#    k_arr = np.arange(N_MELS)
#    n_arr = np.arange(N_MELS)
#    B_raw = np.cos(np.pi / N_MELS * np.outer(k_arr, n_arr + 0.5))
#    B = B_raw.copy()
#    B[0, :] *= ...  # ortho scaling: w0
#    B[1:, :] *= ... # ortho scaling: wk
#    (this B is the same matrix as the D in the earlier "Orthogonal matrices and the inverse transform" section —
#     exactly the same formula, no "variation" to dream up)
#
# 3. Inverse transform (right-multiply by B directly, not B.T — see the derivation in step 3 above)
#    log_mel_recon = dct_truncated @ B
#
# 4. Compute the RMS (root-mean-square) error
#    rms_error = np.sqrt(np.mean((log_mel_recon - log_mel_todo) ** 2))

# ─ Start of implementation ─

# TODO: build the B matrix and implement the inverse transform (don't delete the raise below; "disable" it by commenting it out)

raise NotImplementedError(
    'TODO: complete the four steps above. '
    'Pay special attention to the two ortho scaling weights: w0=1/sqrt(N), wk=sqrt(2/N); '
    'the inverse transform is dct_truncated @ B (not @ B.T)'
)

# Check (uncomment once implemented):
# print(f'reconstruction shape: {log_mel_recon.shape} = {log_mel_todo.shape}')
# rms_error = np.sqrt(np.mean((log_mel_recon - log_mel_todo) ** 2))
# print(f'reconstruction RMS error: {rms_error:.2f} dB')
# assert rms_error < 8.0, f'error {rms_error:.2f} dB is higher than expected (should be < 8 dB; the reference implementation gets about 6 dB)'
# print('✅ the reconstruction error from keeping 13 dims is within the acceptable range')


## Appendix A · Into deep learning: from "forward features" to "backward learning" (required reading, 10 minutes)

1. **What you were doing in L32–L53**: fixed transforms (FFT / mel / MFCC) — audio in, vectors out; **the weights never change**.
2. **What you'll be doing from L54 on**: models with **learnable parameters**; a **loss function** measures how wrong they are; **gradients** tell the parameters which way to move.
3. **The same maths**: L53's DCT is a forward matrix multiplication; L54's `backward()` asks "if the loss changes a little, which parameter should move" — that is, the code version of the L24 chain rule.
4. **No need to learn PyTorch first**: L54–L58 use a hand-written `Value` to understand `loss.backward()`; only at L59 do we switch to PyTorch.
5. **A sentence to keep in mind**: `I learned MFCC to prepare the input for the CNN in L63; I'm learning L54 so the CNN can learn from its mistakes.`

```python
# Feel the difference between "learning" and "features" (nothing to implement)
# features: y = mfcc(x)           — no parameters
# learning: loss = (w*x - y)**2   — need dL/dw
```

## Appendix B · One last look at the MFCC pipeline (back to L50)

`waveform → spectrogram → mel_spectrogram → log → dct_ii → mfcc`

Mnemonic: `(T, 80)` first squeezed to `(T, 40)`, then to `(T, 13)`; after L54 we no longer process audio arrays but scalar computation graphs.

## Wrap-up

This section used `dct_ii` to compress a 40-dim log-mel spectrum into a shorter MFCC, and showed how the variance of the DCT coefficients decays exponentially fast, so the first 13 coefficients hold on to most of the variance.
The `mfcc` function outputs a matrix of shape `(n_frames, n_mfcc)`, each row one audio frame's MFCC vector; c[0] encodes the total log energy, and c[1]/c[2] encode the coarse tilt and curvature of the spectral envelope.
Both functions belong to the `aurora.audio.mfcc` module, and the full chain from log-mel to coefficients is: `mel_spectrogram → power_to_db → dct_ii → keep the first n_mfcc columns`.
The next lesson (**L54**) opens the deep-learning module: implementing an autodiff engine from scratch (Value nodes and computation graphs), so the audio features really get plugged into a training pipeline.

In [ ]:
# ── Standalone maths assertions: checking core properties of the DCT-II (no student implementation needed) ────────────────────────
import numpy as np, sys; sys.path.insert(0, 'src')
from aurora.audio.mfcc import dct_ii
from aurora.audio.mel import mel_spectrogram, power_to_db

# 1. DCT-II orthogonality: D @ D.T = I (ortho normalization)
N = 16
k_arr = np.arange(N); n_arr = np.arange(N)
D = np.cos(np.pi * np.outer(k_arr, 2*n_arr+1) / (2*N))
w0 = 1.0/np.sqrt(N); wk = np.sqrt(2.0/N)
D[0] *= w0; D[1:] *= wk
err_orth = np.max(np.abs(D @ D.T - np.eye(N)))
assert err_orth < 1e-12
print(f"1 ✅  DCT-II orthogonality: max|D·Dᵀ-I|={err_orth:.2e}")

# 2. aurora's dct_ii agrees with the hand-built matrix
x_test = np.random.default_rng(0).standard_normal(N)
ref = D @ x_test
got = dct_ii(x_test, norm='ortho')
assert np.allclose(ref, got, atol=1e-12), f"max diff={np.max(np.abs(ref-got)):.2e}"
print(f"2 ✅  aurora dct_ii(norm='ortho') agrees with the matrix product, max|diff|={np.max(np.abs(ref-got)):.2e}")

# 3. c[0] = mean × √N (the constant basis)
c0_formula = np.mean(x_test) * np.sqrt(N)
assert np.isclose(c0_formula, got[0], atol=1e-12)
print(f"3 ✅  c[0]={got[0]:.6f} = mean(x)·√N = {np.mean(x_test):.6f}·{np.sqrt(N):.4f}")

# 4. Energy compaction: the first 13/40 coefficients of a chirp keep > 80% of the variance
sr, N_mels, N_keep = 22050, 40, 13
from aurora.audio.io import chirp as make_chirp
sig_c = make_chirp(duration=0.5, f0=200, f1=4000, sample_rate=sr)
logM = power_to_db(mel_spectrogram(sig_c, sr, n_fft=1024, n_mels=N_mels), top_db=None)
coeffs = dct_ii(logM, norm='ortho')
var_all = np.var(coeffs, axis=0)
retention = var_all[:N_keep].sum() / var_all.sum() * 100
assert retention > 80.0, f"retention={retention:.1f}%"
print(f"4 ✅  chirp signal: first {N_keep}/{N_mels} coefficients keep {retention:.1f}% of the variance > 80%")

# 5. dct_ii can process batches (2-D)
X_batch = np.random.default_rng(1).standard_normal((10, 13))
C_batch = dct_ii(X_batch, norm='ortho')
assert C_batch.shape == (10, 13)
print(f"5 ✅  batched dct_ii: shape ({10},{13})→{C_batch.shape} (row-by-row transform)")


---

→ **Next lesson** · [L54 · A `Value` computation graph](../6_deep_learning/L54_value_autograd.ipynb)

> **Switching modules**: first read the chain-rule review bridge at the start of L54, then start on the `Value` implementation. If it feels abrupt, come back to Appendix A of this lesson.